#### How to extract `hour, minute and second` from a `timestamp column`?

|   Function    |                      Description                                                                      |
|---------------|-------------------------------------------------------------------------------------------------------|
| **hour**      | Extracts **hour unit** from **Timestamp column or string column** containing a `timestamp`.           |
| **minute**    | Extracts **minute unit** from **Timestamp column or string column** containing a `timestamp`.         |
| **second**    | Extracts the **second unit** from the **Timestamp column or string column** containing a `timestamp`. |

 **Syntax**

      hour(expr)
      minute(expr)
      second(expr)

|  Arguments    |   Description            |
|---------------|--------------------------|
| **expr**      | A TIMESTAMP expression   |
| **Returns**   | INTEGER                  |

**Content:**
- Using Timestamp Column
  - How to Extract hour, minute & second from timestamp column?
  - How to Extract hour, minute & second from string timestamp column?
- Using current_timestamp()
- Using current_timestamp() on dataframe
- Using hour, minute, and second with Filter
- Using hour, minute, and second with GroupBy
- Using SQL Query

In [0]:
from pyspark.sql.functions import col, hour, minute, second, current_date, current_timestamp, to_timestamp, count

##### 1) Using `Timestamp Column`

**a) Extract `hour, minute & second` from `timestamp` column**

**Ex 01:**

In [0]:
data = [["1", "2020-02-01 11:01:19.016"],
        ["2", "2019-03-01 12:01:19.406"],
        ["3", "2021-03-01 12:01:19.406"]]

df_hms_ts = (spark.createDataFrame(data, ["id", "timestamp_str"])
             # Convert String → Timestamp
             .withColumn("timestamp_col", col("timestamp_str").cast("timestamp")))

display(df_hms_ts)
df_hms_ts.printSchema()

id,timestamp_str,timestamp_col
1,2020-02-01 11:01:19.016,2020-02-01T11:01:19.016Z
2,2019-03-01 12:01:19.406,2019-03-01T12:01:19.406Z
3,2021-03-01 12:01:19.406,2021-03-01T12:01:19.406Z


root
 |-- id: string (nullable = true)
 |-- timestamp_str: string (nullable = true)
 |-- timestamp_col: timestamp (nullable = true)



In [0]:
df_hms_ts_final = df_hms_ts.select(col("timestamp_col"),
                                   hour(col("timestamp_col")).alias("hour"),
                                   minute(col("timestamp_col")).alias("minute"),
                                   second(col("timestamp_col")).alias("second"))
              
display(df_hms_ts_final)
df_hms_ts_final.printSchema()

timestamp_col,hour,minute,second
2020-02-01T11:01:19.016Z,11,1,19
2019-03-01T12:01:19.406Z,12,1,19
2021-03-01T12:01:19.406Z,12,1,19


root
 |-- timestamp_col: timestamp (nullable = true)
 |-- hour: integer (nullable = true)
 |-- minute: integer (nullable = true)
 |-- second: integer (nullable = true)



**Ex 02:**

In [0]:
import datetime
from pyspark.sql.functions import col, hour, minute, second

df_dt = spark.createDataFrame([(datetime.datetime(2023, 1, 15, 4, 14, 22),), 
                               (datetime.datetime(2024, 10, 31, 10, 9, 16),),
                               (datetime.datetime(2025, 8, 22, 18, 49, 46),),
                               (datetime.datetime(2021, 9, 19, 22, 55, 26),),
                               (datetime.datetime(2022, 12, 15, 23, 26, 46),),
                               (datetime.datetime(2028, 5, 24, 17, 37, 39),)],
                              ['timestamp_col'])
display(df_dt)

# Extracting the time components
df_extracted = df_dt.select(
    "timestamp_col",
    hour(col("timestamp_col")).alias("hour"),
    minute(col("timestamp_col")).alias("minute"),
    second(col("timestamp_col")).alias("second")
)

# Show the result
display(df_extracted)

timestamp_col
2023-01-15T04:14:22.000Z
2024-10-31T10:09:16.000Z
2025-08-22T18:49:46.000Z
2021-09-19T22:55:26.000Z
2022-12-15T23:26:46.000Z
2028-05-24T17:37:39.000Z


timestamp_col,hour,minute,second
2023-01-15T04:14:22.000Z,4,14,22
2024-10-31T10:09:16.000Z,10,9,16
2025-08-22T18:49:46.000Z,18,49,46
2021-09-19T22:55:26.000Z,22,55,26
2022-12-15T23:26:46.000Z,23,26,46
2028-05-24T17:37:39.000Z,17,37,39


**b) Extract `hour, minute & second` from `string timestamp` column**

In [0]:
data = [["1", "2020-02-01 11:01:19.016"],
        ["2", "2019-03-01 12:01:19.406"],
        ["3", "2021-03-01 12:01:19.406"]]

df_hms_str = spark.createDataFrame(data, ["id", "timestamp_str"])

display(df_hms_str)
df_hms_str.printSchema()

id,timestamp_str
1,2020-02-01 11:01:19.016
2,2019-03-01 12:01:19.406
3,2021-03-01 12:01:19.406


root
 |-- id: string (nullable = true)
 |-- timestamp_str: string (nullable = true)



In [0]:
df_hms_str_final = df_hms_str.select(col("timestamp_str"),
                                     hour(col("timestamp_str")).alias("hour"),
                                     minute(col("timestamp_str")).alias("minute"),
                                     second(col("timestamp_str")).alias("second"))
              
display(df_hms_str_final)
df_hms_str_final.printSchema()

timestamp_str,hour,minute,second
2020-02-01 11:01:19.016,11,1,19
2019-03-01 12:01:19.406,12,1,19
2021-03-01 12:01:19.406,12,1,19


root
 |-- timestamp_str: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- minute: integer (nullable = true)
 |-- second: integer (nullable = true)



##### 2) Using `current_timestamp()`

In [0]:
from pyspark.sql.functions import current_timestamp

df_cts = spark.range(1).select(current_timestamp().alias("current_time"))
display(df_cts)
df_cts.printSchema()

current_time
2026-03-13T18:19:08.344Z


root
 |-- current_time: timestamp (nullable = false)



In [0]:
df_cts_final = df_cts.select("current_time",
                             hour("current_time").alias("hour"),
                             minute("current_time").alias("minute"),
                             second("current_time").alias("second"))

display(df_cts_final)
df_cts_final.printSchema()

current_time,hour,minute,second
2026-03-13T18:19:50.472Z,18,19,50


root
 |-- current_time: timestamp (nullable = false)
 |-- hour: integer (nullable = false)
 |-- minute: integer (nullable = false)
 |-- second: integer (nullable = false)



**3) Using current_timestamp() on dataframe**

In [0]:
df = spark.read.csv("/Volumes/azureadb/pyspark/timestamp/current_date()_current_user()_current_timestamp().csv", header=True, inferSchema=True)
display(df.limit(10))

Commodity_Index,Sensex_Category,Label_Type,Effective_Date,Income_Level,Department,TARGET,Input_Timestamp_UTC
DISCOUNT,Top,average,6-Feb-23,Low,TESTING,SQL Server,1709109264
DISCOUNT,Top,average,6-Feb-23,Low,TESTING,SQL Server,1710234895
DISCOUNT,Top,average,8-Jan-24,Low,TESTING,SQL Server,1709109264
DISCOUNT,Top,average,8-Jan-24,Low,TESTING,SQL Server,1707813327
DISCOUNT,Top,average,6-Mar-23,Low,TESTING,SQL Server,1707813327
DISCOUNT,Forward,medium,6-Mar-23,Low,TESTING,SQL Server,1707813327
DISCOUNT,Forward,medium,6-Jan-25,Low,TESTING,SQL Server,1707813327
DISCOUNT,Forward,medium,6-Jan-25,Low,TESTING,SQL Server,1707813327
DISCOUNT,Forward,medium,6-Apr-23,Low,TESTING,SQL Server,1707813327
DISCOUNT,Forward,medium,6-Apr-23,Low,TESTING,SQL Server,1707813327


In [0]:
df_ct = df.withColumn("updated_datetime", current_timestamp())
display(df_ct.limit(10))
df_ct.printSchema()

Commodity_Index,Sensex_Category,Label_Type,Effective_Date,Income_Level,Department,TARGET,Input_Timestamp_UTC,updated_datetime
DISCOUNT,Top,average,6-Feb-23,Low,TESTING,SQL Server,1709109264,2026-03-13T18:21:16.925Z
DISCOUNT,Top,average,6-Feb-23,Low,TESTING,SQL Server,1710234895,2026-03-13T18:21:16.925Z
DISCOUNT,Top,average,8-Jan-24,Low,TESTING,SQL Server,1709109264,2026-03-13T18:21:16.925Z
DISCOUNT,Top,average,8-Jan-24,Low,TESTING,SQL Server,1707813327,2026-03-13T18:21:16.925Z
DISCOUNT,Top,average,6-Mar-23,Low,TESTING,SQL Server,1707813327,2026-03-13T18:21:16.925Z
DISCOUNT,Forward,medium,6-Mar-23,Low,TESTING,SQL Server,1707813327,2026-03-13T18:21:16.925Z
DISCOUNT,Forward,medium,6-Jan-25,Low,TESTING,SQL Server,1707813327,2026-03-13T18:21:16.925Z
DISCOUNT,Forward,medium,6-Jan-25,Low,TESTING,SQL Server,1707813327,2026-03-13T18:21:16.925Z
DISCOUNT,Forward,medium,6-Apr-23,Low,TESTING,SQL Server,1707813327,2026-03-13T18:21:16.925Z
DISCOUNT,Forward,medium,6-Apr-23,Low,TESTING,SQL Server,1707813327,2026-03-13T18:21:16.925Z


root
 |-- Commodity_Index: string (nullable = true)
 |-- Sensex_Category: string (nullable = true)
 |-- Label_Type: string (nullable = true)
 |-- Effective_Date: string (nullable = true)
 |-- Income_Level: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- TARGET: string (nullable = true)
 |-- Input_Timestamp_UTC: integer (nullable = true)
 |-- updated_datetime: timestamp (nullable = false)



In [0]:
df_ct = df_ct.select("updated_datetime")\
             .withColumn("hour", hour(df_ct.updated_datetime))\
             .withColumn("minute", minute(df_ct.updated_datetime))\
             .withColumn("second", second(df_ct.updated_datetime))

display(df_ct.limit(10))
df_ct.printSchema()

updated_datetime,hour,minute,second
2026-02-15T06:27:32.157Z,6,27,32
2026-02-15T06:27:32.157Z,6,27,32
2026-02-15T06:27:32.157Z,6,27,32
2026-02-15T06:27:32.157Z,6,27,32
2026-02-15T06:27:32.157Z,6,27,32
2026-02-15T06:27:32.157Z,6,27,32
2026-02-15T06:27:32.157Z,6,27,32
2026-02-15T06:27:32.157Z,6,27,32
2026-02-15T06:27:32.157Z,6,27,32
2026-02-15T06:27:32.157Z,6,27,32


root
 |-- updated_datetime: timestamp (nullable = false)
 |-- hour: integer (nullable = false)
 |-- minute: integer (nullable = false)
 |-- second: integer (nullable = false)



##### 4) Using hour, minute, and second with Filter

In [0]:
# Create a DataFrame with a timestamp column
data = [("2022-07-25 12:30:45",),
        ("2022-07-25 12:30:45",),
        ("2023-01-01 08:45:00",),
        ("2022-12-31 23:59:59",),
        ("2022-01-01 00:00:00",),
        ("2022-06-01 14:30:00",),
        ("2022-06-01 14:30:00",),
        ("2022-06-01 14:30:00",)]

df_fltr = spark.createDataFrame(data, ["timestamp"])

# Convert the timestamp column to a TimestampType
df_fltr = df_fltr.withColumn("timestamp", to_timestamp("timestamp"))
display(df_fltr)

timestamp
2022-07-25T12:30:45.000Z
2022-07-25T12:30:45.000Z
2023-01-01T08:45:00.000Z
2022-12-31T23:59:59.000Z
2022-01-01T00:00:00.000Z
2022-06-01T14:30:00.000Z
2022-06-01T14:30:00.000Z
2022-06-01T14:30:00.000Z


In [0]:
# Extract the hour, minute, and second from the timestamp column
df_fltr_add = df_fltr \
    .withColumn("hour", hour(col("timestamp"))) \
    .withColumn("minute", minute(col("timestamp"))) \
    .withColumn("second", second(col("timestamp")))

display(df_fltr_add)

timestamp,hour,minute,second
2022-07-25T12:30:45.000Z,12,30,45
2022-07-25T12:30:45.000Z,12,30,45
2023-01-01T08:45:00.000Z,8,45,0
2022-12-31T23:59:59.000Z,23,59,59
2022-01-01T00:00:00.000Z,0,0,0
2022-06-01T14:30:00.000Z,14,30,0
2022-06-01T14:30:00.000Z,14,30,0
2022-06-01T14:30:00.000Z,14,30,0


In [0]:
# Filter for rows where the hour is between 8 and 12
df_fltr_final = df_fltr_add.filter(col("hour").between(8, 12))

# Display the result
display(df_fltr_final)

timestamp,hour,minute,second
2022-07-25T12:30:45.000Z,12,30,45
2022-07-25T12:30:45.000Z,12,30,45
2023-01-01T08:45:00.000Z,8,45,0


##### 5) Using hour, minute, and second with GroupBy

In [0]:
# Group by hour, minute, and second and count the number of rows
df_grp = df_fltr_add \
    .groupBy("hour", "minute", "second").agg(count("*").alias("count"))

# Display the result
display(df_grp)

hour,minute,second,count
12,30,45,2
8,45,0,1
23,59,59,1
0,0,0,1
14,30,0,3


##### 6) Using SQL Query

In [0]:
df_cts.createOrReplaceTempView("time_table")

spark.sql("""
SELECT 
    current_time,
    hour(current_time)   AS hour,
    minute(current_time) AS minute,
    second(current_time) AS second
FROM time_table
""").display()

current_time,hour,minute,second
2026-03-13T18:32:23.915Z,18,32,23


In [0]:
%sql
SELECT 
    current_time,
    hour(current_time)   AS hour,
    minute(current_time) AS minute,
    second(current_time) AS second
FROM time_table;

current_time,hour,minute,second
2026-03-13T18:33:12.421Z,18,33,12
